In [ ]:
# Copyright 2026 International Business Machines

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

#     http://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Workshop 1.1 Introduction to EO Foundation Models

## Fine-Tuning with TerraMind

---

## 📚 Workshop Overview

This workshop introduces participants to **Earth Observation (EO) foundation models** and provides hands-on experience with fine-tuning and inference workflows.

In this notebook you will work with [TerraMind](https://huggingface.co/ibm-esa-geospatial) to:

- Understand EO foundation model concepts
- Perform a simple fine-tuning workflow 
- Run inference with pre-trained TerraMind models
- Compare single-modal (Sentinel-2) vs multi-modal (RGB+DEM) predictions

The session builds practical intuition for how EO foundation models are adapted to downstream tasks.  
  
Please see the Hugging Face page for more details of the saltmarsh models used in this session.

---

## 📋 Prerequisites

Please read the [instructions](https://github.com/IBM/ML4EO-workshop-2026) on the repository landing page.

---

## 0. Setup

### 0.1 Check Python Version

It's recommended that you run this notebook using Python 3.12.

In [ ]:
!python --version

### 0.2 Install and Import Dependencies

If running locally, ensure you have the necessary packages installed. This assumes you're in the project root directory.

Once installed, import dependencies.

In [ ]:
# Install dependencies 
import sys
if "google.colab" in sys.modules:
    !git clone https://github.com/IBM/ML4EO-workshop-2026.git
    %cd /content/ML4EO-workshop-2026/workshop/1_Introduction_to_EO_FMs 
    !uv pip install --system gdown huggingface_hub terratorch
else:
    print("""
  If you haven't set up your virtual environment already, you can...
  1. go to the same directory as the pyproject.toml file using the command line
  2. run "uv sync" in your command line. This will create a virtual environment with the relevant packages installed. 
  3. select the Python interpreter from the virtual environment 
     (typically located at '.venv/bin/python') as your notebook kernel in VS Code or Jupyter to follow through the examples in this notebook
  """)


import the relevant packages

In [ ]:
from pathlib import Path
import warnings
import os

from helper import (
    download_and_extract_zip,
    download_model_from_hf,
    locate_checkpoints,
    plot_s2_model_pred,
    plot_s2_rgbdem_model_pred,
)

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("✓ Dependencies imported successfully!")


---

## 1. Fine-Tuning

### 1.1 Introduction to Fine-Tuning

Fine-tuning is the process of adapting a pre-trained foundation model to a specific task or domain. This approach leverages the knowledge learned during pre-training while specializing the model for your use case, enabling a task-specific model to be created.


In this section, we'll fine-tune a TerraMind model on salt marsh extent detection using Sentinel-2 imagery.

### 1.2 Download Sample Dataset

Before proceeding, you'll need to download the **sample dataset** for fine-tuning.

The dataset will be downloaded from Google Drive as a zip file and extracted to the `data/` directory.

Expected structure after extraction:
```
data/saltmarsh_sample/
  ├── train/
  │   ├── images/
  │   └── labels/
  ├── val/
  │   ├── images/
  │   └── labels/
  └── test/
      ├── images/
      └── labels/
```

In [ ]:
file_id = "1gLiN7H0Ue-PG7GaFLpOTS8EKVcLRpn6r"

# Download and extract the dataset
download_and_extract_zip(
    file_id=file_id,
    zip_filename="saltmarsh_sample.zip",
    extract_to="../../data",
)

### 1.3 Prepare Fine-Tuning Command

We'll use TerraTorch's command-line interface to fine-tune the model. The configuration file (local, in the repo) specifies all the training parameters.

For this demo, we'll:
- Train for **1 epoch** only
- Use the **sample dataset** (smaller subset for demonstration)

The model will be trained using the local config file which contains all the hyperparameters and model architecture details.

The fine-tuning process is carried out by using the data defined under 

`train_data_root`

`train_label_data_root` 

`val_data_root` 

`val_label_data_root`


In [ ]:
# Project root (adjust if running from different location, say, if you're running locally in your terminal)
project_root = Path("../..")
print(f"Project root: {project_root}")

# Local config file for fine-tuning (in repo)
config_file = project_root / "configs/finetune/config-ibm-geospatial-saltmarsh-uk-s2-extent-blank.yaml"

# Construct the fine-tuning command
fine_tuning_command = f"""terratorch fit \
    --config {config_file} \
"""

print("Fine-tuning command:")
print(fine_tuning_command)


### 1.4 Run Fine-Tuning

Execute the fine-tuning command. This will train the model for 1 epoch and save checkpoints to the output directory.

**Note:** TerraTorch (built on PyTorch Lightning) automatically creates the directory structure:
- `lightning_logs/` - Training logs directory
- `version_0/` - First run (increments for subsequent runs)
- `checkpoints/` - Contains the `.ckpt` checkpoint files

### Exercise: 

Adapt the [example config](../../configs/finetune/config-ibm-geospatial-saltmarsh-uk-s2-extent-blank.yaml) `config-ibm-geospatial-saltmarsh-uk-s2-extent-blank.yaml` to fine-tune a 6 band sentinel 2 model that will classify if a given pixel is a saltmarsh or not. The bands of interest are: BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2. For the purposes of this notebook please set up your model to run a maximum of one epoch. Use the cells below to run fine-tuning. Check that you have a checkpoint written out. 



<details>
<summary><b>Solution</b></summary>
You can get the correct fine-tuning set-up by running the following in a code cell. 

```python
# solution config 
config_file = (
    project_root
    / "configs/finetune/config-ibm-geospatial-saltmarsh-uk-s2-extent-solution.yaml"
)  

# Construct the fine-tuning command that uses the solution config
fine_tuning_command = f"""terratorch fit \
    --config {config_file} \
"""



In [ ]:
config_file = (
    project_root
    / "configs/finetune/config-ibm-geospatial-saltmarsh-uk-s2-extent-blank.yaml"
)  

fine_tuning_command = f"""terratorch fit \
    --config {config_file} \
"""

fine_tuning_command

In [ ]:
# imposing max_epoch = 1 for testing purposes in case it's not defined in the config
fine_tuning_command_with_one_epoch_batch_two = f"{fine_tuning_command}\
    --trainer.max_epochs 1 \
    --data.init_args.batch_size 2 \
    "

print(f"Will now run '{fine_tuning_command_with_one_epoch_batch_two}'")

print("\nNote: Using max_epochs=1 and batch_size=2 for quick demo.")
print(
    "For full training, increase epochs and adjust batch size based on available GPU memory."
)


In [ ]:
# Run fine-tuning
!{fine_tuning_command_with_one_epoch_batch_two}

### 1.5 Locate the New Checkpoint

After training, let's find the checkpoint that was just created.

In [ ]:
output_dir = project_root / "data/finetune/s2" 

new_checkpoint_dir = locate_checkpoints(output_dir)

# Use the first checkpoint for subsequent steps if needed
new_checkpoint_path = new_checkpoint_dir[0]
print(f"\nWill use first checkpoint found: {new_checkpoint_path}")

---

## 2. Testing

### 2.1 Run Test on Fine-Tuned Model

Now we'll test the model performance on the test dataset.

This will evaluate the model's performance on held-out test data and report metrics.

We can use the checkpoint that you just fine-tuned for a single epoch, but let's grab a fully fine-tuned version of the model from Hugging Face. 

In [ ]:
s2_repo_id = "rosielickorish/ibm-geospatial-saltmarsh-uk-s2-extent-10m"
s2_config_name = "config-ibm-geospatial-saltmarsh-uk-s2-extent.yaml"
s2_checkpoint_name = "ibm-geospatial-saltmarsh-uk-s2-extent-10m_state_dict.ckpt"

print("Downloading S2 model from HuggingFace...")
_, s2_checkpoint_path = download_model_from_hf(
    repo_id=s2_repo_id,
    config_name=s2_config_name,
    checkpoint_name=s2_checkpoint_name,
    project_root=project_root,
)

print(f"✓ S2 checkpoint: {s2_checkpoint_path}")

Terratorch will compute metrics using data that it finds in the config's `test_data_root` and `test_label_data_root`

### Exercise: 
Review what is under `test_data_root` and `test_label_data_root` then run the test below. How many images and labels are you testing on? 


In [ ]:
# Construct the test command using the newly fine-tuned checkpoint
test_command = f"""terratorch test \
    --config {config_file} \
    --ckpt_path {s2_checkpoint_path} \
"""

print("Test command:")
print(test_command)

In [ ]:
# Run testing
!{test_command}

---

## 3. Inference with Pre-Trained Models


### 3.1 Download Inference Data

Download the inference dataset from Google Drive. This contains sample images for running inference with both S2 and RGB+DEM models. The sample images are that of the Dee marsh. 

Expected structure after extraction:
```
data/inference/
  └── images/
      ├── s2/          # Sentinel-2 imagery
      └── aerial_extent/      # RGB + DEM aerial imagery
```

N.B. Terratorch works by looking for files with the same prefix to identify image-label pairs. The downloaded marsh labels are from 2018 but have been renamed to 2024 so that it matches the imagery year to allow terratorch to find these pairs. 

In [ ]:

inference_file_id = "1Wp4aZf75zy_kErHnwlXCNqWjXoQd5a-8"

extract_dir = "../../data/inference/images"
os.makedirs(extract_dir, exist_ok=True)

# Download 
download_and_extract_zip(
    file_id=inference_file_id,
    zip_filename="inference_data.zip",
    extract_to=extract_dir,
)

### 3.2 Run Inference with Sentinel-2 Model

Run prediction on inference images using the pre-trained S2 checkpoint.

Terratorch requires four things to run inference:

`path to config`

`path to model checkpoint`

`path to inference input`

`path to inference output`

We've already defined the first two items in the exercise above, let's define the last two items.



### Exercise: 

Find the inference input in your downloaded data, run inference, then plot the output using the cells below. 

__Extension__: Can you work out the performance metrics of the predictions on the Dee marsh using the `terratorch test ...` command? 


In [ ]:
# let's define the last two items out of the four that terratorch expects
inference_images_s2 = project_root / "..." # to fill in 
s2_inference_output = project_root / "data/inference/pred/s2" # do not change this for the purposes of this demo notebook 

<details>
<summary><b>Solution</b></summary>

```python
inference_images_s2 = project_root / "data/inference/images/s2"




In [ ]:
# Use the downloaded config from HuggingFace
# Construct the prediction command
s2_predict_command = f"""terratorch predict \
    --config {config_file} \
    --ckpt_path {s2_checkpoint_path} \
    --trainer.default_root_dir {output_dir} \
    --predict_output_dir {s2_inference_output} \
    --data.init_args.predict_data_root.S2L2A {inference_images_s2}"""

print("S2 Prediction command:")
print(s2_predict_command)
print(f"\nUsing config: {config_file}")
print(f"Using checkpoint: {s2_checkpoint_path}")

In [ ]:
# Run S2 prediction
!{s2_predict_command}

Let's visualize the predictions from the S2 model alongside the input imagery and labels.

In [ ]:
# find the predicted file
s2_pred_file = list(s2_inference_output.glob("*pred*"))
print(s2_pred_file)

In [ ]:
# explicitly defining the files for the helper function to work 
s2_img_file = project_root / "data/inference/images/s2/ortho_s2_dee_2024_res_10m_img.tif"
s2_lab_file = project_root / "data/inference/images/s2/ortho_s2_dee_2024_res_10m_lab.tif"

In [ ]:
# Visualize S2 model predictions
plot_s2_model_pred(s2_img_file, s2_lab_file, s2_pred_file)

<details>
<summary><b>Extension Activity Solution</b></summary>

Run the following in a code cell.

```python 
# Dee marsh performance metrics 
test_command = f"""terratorch test \
    --config {config_file} \
    --ckpt_path {s2_checkpoint_path} \
    --data.init_args.test_data_root.S2L2A ../../data/inference/images/s2/ \
    --data.init_args.test_label_data_root ../../data/inference/images/s2/ \
"""

print("Now running test command:")
print(test_command)

!{test_command}


---

## 4. Multi-Modal Inference

### 4.1 Run Inference with RGB+DEM Model

Now let's run inference using a multi-modal model that combines aerial RGB imagery with Digital Elevation Model (DEM) data.

This model uses:
- **RGB**: High-resolution aerial imagery (2m resolution)
- **DEM**: Digital Elevation Model for terrain information

We'll use the **config from HuggingFace** for this model.

In [ ]:
# get the HF checkpoint and config 
rgbdem_repo_id = "rosielickorish/ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m"
rgbdem_config_name = "config-ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m.yaml"
rgbdem_checkpoint_name = "ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m_state_dict.ckpt"

print("\nDownloading RGB+DEM model from HuggingFace...")
rgbdem_config_path, rgbdem_checkpoint_path = download_model_from_hf(
    repo_id=rgbdem_repo_id,
    config_name=rgbdem_config_name,
    checkpoint_name=rgbdem_checkpoint_name,
    project_root=project_root,
)
print(f"✓ RGB+DEM config: {rgbdem_config_path}")
print(f"✓ RGB+DEM checkpoint: {rgbdem_checkpoint_path}")

### Exercise: 

Examine the newly downloaded config from Hugging Face. How are the additional modalities defined?  

Define the paths needed for inference in the cell below so that you can get predictions using this model for the Dee marsh. 

You can use the cells below to run inference and plot the new predictions. 

__Extension__: Can you work out the performance metrics of the predictions on the Dee marsh using the `terratorch test ...` command? 

In [ ]:
rgbdem_output_dir = ... # where do you want to save your run logs? 
rgbdem_inference_output = ... #where do you want to save your predictions?  
inference_images_aerial = ... #where are the aerial imagery you want to use for inference? 
inference_images_dem = ... #where are the elevation data you want to use for inference? 


<details>
<summary><b>Example Solution</b></summary>
Run the following in a code cell. 

```python

rgbdem_output_dir = project_root / "data/inference/images/pred/logs" # up to the user
rgbdem_inference_output = project_root / "data/inference/images/pred/" # up to the user
inference_images_aerial = project_root / "data/inference/images/aerial_extent" 
inference_images_dem = project_root / "data/inference/images/aerial_extent" 


In [ ]:
# Use the downloaded config from HuggingFace
# Construct the prediction command for RGB+DEM model
rgbdem_predict_command = f"""terratorch predict \
    --config {rgbdem_config_path} \
    --ckpt_path {rgbdem_checkpoint_path} \
    --trainer.default_root_dir {rgbdem_output_dir} \
    --predict_output_dir {rgbdem_inference_output} \
    --data.init_args.predict_data_root.RGB {inference_images_aerial} \
    --data.init_args.predict_data_root.DEM {inference_images_dem}"""

print("RGB+DEM Prediction command:")
print(rgbdem_predict_command)
print(f"\nUsing config: {rgbdem_config_path}")
print(f"Using checkpoint: {rgbdem_checkpoint_path}")

In [ ]:
# Run RGB+DEM prediction
!{rgbdem_predict_command}

In [ ]:
# find the prediction
rgbdem_pred_file = list(rgbdem_inference_output.glob("*pred*"))
print(rgbdem_pred_file)

### 4.2 Compare S2 vs RGB+DEM Models

Let's compare the predictions from both models to understand the benefits of multi-modal approaches.

We'll create a 2-row comparison showing:
- **Row 1 (S2 Model)**: Input imagery, DEM (blank), Labels, Predictions
- **Row 2 (RGB+DEM Model)**: Input imagery, DEM, Labels, Predictions

In [ ]:
# explicitly define these for the helper function to work 
rgbdem_img_file = project_root / "data/inference/images/aerial_extent/ortho_RGBN_dee_2024_res_2m_img.tif"
rgbdem_dem_file = project_root / "data/inference/images/aerial_extent/ortho_RGBN_dee_2024_res_2m_dtm.tif"
rgbdem_lab_file = project_root / "data/inference/images/aerial_extent/ortho_RGBN_dee_2024_res_2m_lab.tif"

In [ ]:
# Visualize comparison of S2 and RGB+DEM model predictions
plot_s2_rgbdem_model_pred(
    s2_img_file,
    s2_lab_file,
    s2_pred_file,
    rgbdem_img_file,
    rgbdem_dem_file,
    rgbdem_lab_file,
    rgbdem_pred_file,
)

<details>
<summary><b>Extension Activity Solution</b></summary>

Run the following in a code cell. 

```python 
# Dee marsh performance metrics 
test_command = f"""terratorch test \
    --config {rgbdem_config_path} \
    --ckpt_path {rgbdem_checkpoint_path} \
    --data.init_args.test_data_root.RGB ../../data/inference/images/aerial_extent/ \
    --data.init_args.test_data_root.DEM ../../data/inference/images/aerial_extent/ \
    --data.init_args.test_label_data_root ../../data/inference/images/aerial_extent/ \
"""

print("Test command:")
print(test_command)

!{test_command}

### Key Insights: Multi-Modal vs Single-Modal

Comparing the two models reveals important differences:

**Multi-modal (RGB+DEM) advantages:**
- **Higher Resolution**: Aerial imagery at 2m vs Sentinel-2 at 10m provides finer spatial detail
- **Terrain Context**: DEM provides elevation information crucial for salt marsh identification
- **Complementary Information**: RGB captures visual features while DEM captures topographic features
- **Improved Accuracy**: Combining modalities typically improves detection performance in complex areas

**Single-modal (S2) advantages:**
- **Global Coverage**: Sentinel-2 provides consistent global coverage
- **Temporal Resolution**: Regular revisit times (5 days) for monitoring changes over time
- **Free and Open**: Publicly available data with no acquisition costs
- **Spectral Bands**: Multiple spectral bands beyond RGB (NIR, SWIR, etc.) for vegetation analysis
- **Scalability**: Easier to scale to large areas without custom aerial surveys

**When to use each:**
- Use **S2 models** for: Large-area mapping, temporal monitoring, global applications, budget-constrained projects
- Use **RGB+DEM models** for: High-precision mapping, detailed site assessments, areas with complex topography

### Key Insights: Multi-Modal vs Single-Modal

Multi-modal models (RGB+DEM) often provide advantages:

- **Higher Resolution**: Aerial imagery at 2m vs Sentinel-2 at 10m
- **Terrain Context**: DEM provides elevation information crucial for salt marsh identification
- **Complementary Information**: RGB captures visual features while DEM captures topographic features
- **Improved Accuracy**: Combining modalities typically improves detection performance

However, single-modal S2 models have benefits too:

- **Global Coverage**: Sentinel-2 provides consistent global coverage
- **Temporal Resolution**: Regular revisit times for monitoring changes
- **Free and Open**: Publicly available data
- **Spectral Bands**: Multiple spectral bands beyond RGB

---

## 📖 Next Steps

Continue to the next notebook:

- **1.2 Multimodal Inference with TerraMind** - Learn more about running inference with different data modalities

---

## 🔗 Additional Resources

- [TerraMind on HuggingFace](https://huggingface.co/ibm-esa-geospatial)
- [TerraTorch Documentation](https://github.com/IBM/terratorch)
- [TerraKit Documentation](https://torchgeo.org/terrakit/)
- [Prithvi Model Hub](https://huggingface.co/ibm-nasa-geospatial)

---

<!--
Copyright (c) 2026 IBM Corp

This software is released under the MIT License.
https://opensource.org/licenses/MIT
-->